# vLLM 커널 실습 — group quantization (Triton)

vLLM의 CUDA quantization 커널(`per_token_group_quant_8bit_kernel`)을 Triton으로 재구현하고, `torch.ops`에 커스텀 op으로 등록해보는 실습 노트북입니다.

**먼저 런타임을 GPU로 설정하세요**: 메뉴 → 런타임 → 런타임 유형 변경 → GPU

## 1. 이게 뭐을 하는 코드인가

**목표**: 큰 텐서를 통째로 다루지 않고, `group_size`개씩 묶은 "그룹" 단위로 각각 다른 scale을 적용해서 8bit(int8)로 압축한다. LLM 추론에서 이렇게 하는 이유:

- 모델 가중치/activation을 fp16 → int8로 줄이면 메모리 대역폭을 4배 아낌벼 수 있음 (fp16 2바이트 → int8 1바이트, 게다가 scale은 그룹당 하나라 거의 무시할 크기)
- 텐서 전체에 scale 하나만 쓰면(per-tensor quant) 값 분포가 다른 구간에서 정밀도가 깨짐 → 작은 그룹 단위로 쪼개서(`group_size`개씩) 그룹마다 다른 scale을 쓰면 정밀도 손실을 줄일 수 있음 (이게 vLLM 원본 커널 `per_token_group_quant_8bit_kernel`의 존재 이유)
- Edge 디바이스처럼 메모리 대역폭이 병목인 환경에서 특히 효과가 큼

수식은 그룹 하나마다:
```
absmax = max(|x_1|, ..., |x_n|)          # 그룹 안에서 절대값 최대값
scale  = absmax / 127                     # int8 표현 범위(-127~127)에 맞춰 스케일
q_i    = clamp(x_i / scale, -127, 127)     # 스케일로 나눐서 8bit 정수화
```

In [ ]:
import torch
import triton
import triton.language as tl

print("torch:", torch.__version__, "| triton:", triton.__version__)
assert torch.cuda.is_available(), "런타임을 GPU로 바꿔주세요 (런타임 > 런타임 유형 변경 > GPU)"

## 2. Triton 커널 뜼어보기

```python
@triton.jit
def _group_quant_int8_kernel(...):
    pid = tl.program_id(axis=0)
```
- Triton은 커널을 grid 위에 병렬로 뿌려서 실행한다. `pid`는 "지금 이 인스턴스가 grid에서 몇 번째냐"를 알려주는 값 — CUDA의 `blockIdx.x`랑 같은 개념.
- 이 커널은 **`pid` 하나 = 그룹 하나**로 설계했다. 그래서 `grid = (num_tokens * num_groups,)`로 런칭하면, 전체 텐서의 모든 그룹이 동시에(논리적으로) 처리됨.

```python
    num_groups = hidden_size // group_size
    token_id = pid // num_groups
    group_id = pid % num_groups
```
- `pid`라는 1차원 번호를 다시 (몇 번째 토큰, 몇 번째 그룹)이라는 2차원 좌표로 푸는 부분.

```python
    offs = tl.arange(0, group_size)
    row_start = token_id * hidden_size + group_id * group_size
    x = tl.load(x_ptr + row_start + offs).to(tl.float32)
```
- `tl.arange(0, group_size)`: `[0, 1, ..., group_size-1]` — 그룹 안에서의 상대 위치들.
- `tl.load(...)`: 실제 GPU 메모리(DRAM)에서 `group_size`개 원소를 한 번에 벡터로 읽어옵니다.

```python
    absmax = tl.max(tl.abs(x), axis=0)
    scale = absmax / 127.0
    scale = tl.where(scale == 0.0, 1.0, scale)
```
- `tl.max(..., axis=0)`: 방금 읽은 `group_size`개 값 중 최대값 하나로 리덕션(reduce). Triton이 내부적으로 병렬 리덕션을 알아서 컴파일해줌 — CUDA로 직접 짜면 워프 셔플/shared memory 리덕션을 손으로 짜야 하는 부분.
- 그룹 전체가 0이면 0으로 나누기를 막기 위해 `tl.where`로 그 경우만 1.0으로 치환.

```python
    q = tl.minimum(tl.maximum(x / scale, -127.0), 127.0)
    tl.store(out_q_ptr + row_start + offs, q.to(tl.int8))
    tl.store(out_s_ptr + token_id * num_groups + group_id, scale)
```
- `q.to(tl.int8)`: float → int8 캐스트. **반올림이 아니라 truncate(0 방향으로 자름)** — 이건 원본 vLLM CUDA 커널(`DST_DTYPE(q)` 캐스트)과 동일하게 맞춘 부분.

In [ ]:
@triton.jit
def _group_quant_int8_kernel(
    x_ptr,          # *input, shape [num_tokens, hidden_size], row-major
    out_q_ptr,      # *output int8, same shape as x
    out_s_ptr,      # *output fp32 scales, shape [num_tokens, num_groups]
    hidden_size,    # int
    group_size: tl.constexpr,
):
    # One program == one (token, group) pair.
    pid = tl.program_id(axis=0)
    num_groups = hidden_size // group_size
    token_id = pid // num_groups
    group_id = pid % num_groups

    offs = tl.arange(0, group_size)
    row_start = token_id * hidden_size + group_id * group_size

    x = tl.load(x_ptr + row_start + offs).to(tl.float32)

    absmax = tl.max(tl.abs(x), axis=0)
    scale = absmax / 127.0
    scale = tl.where(scale == 0.0, 1.0, scale)  # avoid div-by-zero on all-zero groups

    # vLLM's reference kernel does a plain truncating cast here (no
    # rounding) - `dst = DST_DTYPE(q)` in per_token_group_quant.cu - so we
    # match that instead of rounding to nearest.
    q = tl.minimum(tl.maximum(x / scale, -127.0), 127.0)

    tl.store(out_q_ptr + row_start + offs, q.to(tl.int8))
    tl.store(out_s_ptr + token_id * num_groups + group_id, scale)


def group_quant_int8_triton(x: torch.Tensor, group_size: int):
    """Quantize the last dim of `x` in chunks of `group_size` to int8."""
    assert x.is_cuda, "Triton 커널은 CUDA 텐서가 필요합니다 (Colab GPU 런타임 사용)"
    assert x.ndim == 2
    num_tokens, hidden_size = x.shape
    assert hidden_size % group_size == 0

    num_groups = hidden_size // group_size
    out_q = torch.empty_like(x, dtype=torch.int8)
    out_s = torch.empty((num_tokens, num_groups), dtype=torch.float32, device=x.device)

    grid = (num_tokens * num_groups,)
    _group_quant_int8_kernel[grid](
        x, out_q, out_s, hidden_size, group_size=group_size
    )
    return out_q, out_s


# 빠른 동작 확인
x_demo = torch.randn(2, 8, device="cuda", dtype=torch.float16) * 5
q_demo, s_demo = group_quant_int8_triton(x_demo, group_size=4)
print("x:", x_demo)
print("q:", q_demo)
print("s:", s_demo)

## 3. PyTorch 연동 (`torch.library.custom_op`)

```python
@torch.library.custom_op("practice::group_quant_int8", mutates_args=())
def group_quant_int8(x, group_size) -> list[torch.Tensor]:
    q, s = group_quant_int8_triton(x, group_size)
    return [q, s]
```
- `torch.library.custom_op`는 이 함수를 PyTorch 연산자(operator)로 등록해준다. 등록하고 나면 `torch.ops.practice.group_quant_int8(...)`로 호출 가능해지고, `torch.compile`/autograd 같은 PyTorch 기능들이 이걸 "블랙박스 연산"으로 인식해서 올바르게 다룰 수 있게 됨.
- vLLM은 이걸 C++로 한다 (`csrc/libtorch_stable/torch_bindings.cpp`의 `STABLE_TORCH_LIBRARY_FRAGMENT` + `ops.def("...")` + `ops.impl(...)`). Python `custom_op` 데코레이터는 그 두 단계(스키마 선언 + 구현 바인딩)를 한 번에 해주는 축약판이라고 보면 됨.

```python
@group_quant_int8.register_fake
def _(x, group_size) -> list[torch.Tensor]:
    ...
```
- "fake" 구현: 실제로 커널을 돌리지 않고 **출력 텐서의 shape/dtype만** 계산해서 반환. `torch.compile`이 그래프를 미리 분석(트레이싱)할 때, 실제 GPU 연산 없이 이 함수로 "이 연산의 출력이 어떤 모양일지"만 알아내는 용도.

In [ ]:
@torch.library.custom_op("practice::group_quant_int8", mutates_args=())
def group_quant_int8(x: torch.Tensor, group_size: int) -> list[torch.Tensor]:
    q, s = group_quant_int8_triton(x, group_size)
    return [q, s]


@group_quant_int8.register_fake
def _(x: torch.Tensor, group_size: int) -> list[torch.Tensor]:
    num_tokens, hidden_size = x.shape
    num_groups = hidden_size // group_size
    q = x.new_empty(x.shape, dtype=torch.int8)
    s = x.new_empty((num_tokens, num_groups), dtype=torch.float32)
    return [q, s]


# 등록되었는지 torch.ops.practice.* 로 호출해서 확인
x = torch.randn(4, 8, device="cuda", dtype=torch.float16)
q, s = torch.ops.practice.group_quant_int8(x, group_size=4)
print("q:", q)
print("s:", s)

## 4. 디버깅하면서 실제로 겦은 버그 두 개

### 버그 1 — 반올림 방식이 서로 달랐다
처음엔 Triton 커널은 "0.5는 무조건 멀리 반올림"(round-half-away-from-zero)으로 짜고, PyTorch reference는 `.round()`(0.5는 짝수 쪽으로 반올림, banker's rounding)를 그대로 쓐음. 두 방식은 딱 `x.5`처럼 정확히 경계에 걸린 값에서만 다른 답을 내. `raw=-126.5`인 원소 하나에서 실제로 걸림 (`-126` vs `-127`).

→ **알아낸 것**: 원본 vLLM 커널은 애초에 반올림을 안 하고 그냥 truncate만 한다(`dst = DST_DTYPE(q)`, C++ 캐스트). 그래서 커널과 reference 둘 다 truncate로 맞춰서 해결.

### 버그 2 — Triton과 PyTorch의 나눗셈이 bit-identical하지 않다
반올림을 통일한 뒤에도 9472개 중 20개가 여전히 갈렸음. 값을 첍어보니 전부 `raw`가 `126.999996`, `-127.000002`처럼 **정수 바로 근처**였음.

→ **원인**: GPU는 나눗셈(`/`)을 성능 때문에 근사 역수(fast reciprocal approximation)로 계산하는 경우가 많아서, Triton의 `x/scale`과 PyTorch의 `x/scale`이 수학적으로 같은 식이어도 **최하위 비트 수준에서 다른 결과**를 낼 수 있음. 그 미세한 오차가 하필 정수 경계를 넘나들면(126.9999961 vs 127.0000004), truncate 결과가 이웃한 정수로 갈라짐.

→ **고친 방법**: "완전 일치"를 요구하는 대신, "경계 근처에서만, 1% 미만으로, 최대 1 차이까지"는 정상으로 보고 통과시키도록 테스트를 바꿈. 이게 실제로 GPU 커널 테스트 코드들이 흔히 쓰는 방식 (bit-exact가 아니라 tolerance 기반 검증).

**왜 이게 중요한 이야기인가**: "정밀도 vs 속도" 트레이드오프는 edge 디바이스(HyperAccel의 도메인)에서 매일 부딚힐는 문제. fast-math를 켜면 빠르지만 근사 오차가 생기고, 정밀한 연산을 쓰면 느려짐. 이 작은 실습에서 그 트레이드오프를 직접 눈으로 본 셔이다.

In [ ]:
def reference_group_quant_int8(x: torch.Tensor, group_size: int):
    """Pure-PyTorch reference: 같은 absmax-quantization 수식, Triton 없이."""
    num_tokens, hidden_size = x.shape
    num_groups = hidden_size // group_size
    xg = x.reshape(num_tokens, num_groups, group_size).float()

    absmax = xg.abs().amax(dim=-1)
    scale = (absmax / 127.0).clamp_min(1e-12)

    # truncating cast (반올림 없음), vLLM의 `DST_DTYPE(q)` 캐스트와 동일
    q = (xg / scale.unsqueeze(-1)).clamp(-127, 127).to(torch.int8)
    return q.reshape(num_tokens, hidden_size), scale


# 버그 2를 직접 눈으로 확인: 경계값 근처에서만 갈림
torch.manual_seed(0)
x_big = torch.randn(37, 256, device="cuda", dtype=torch.float16) * 5
group_size = 64

q_triton, s_triton = group_quant_int8_triton(x_big, group_size)
q_ref, s_ref = reference_group_quant_int8(x_big, group_size)

diff = (q_triton.int() - q_ref.int()).abs()
num_mismatched = (diff != 0).sum().item()
print(f"{num_mismatched} / {q_triton.numel()} 원소가 다름 (경계값 근처 fp32 나눗셈 오차)")
print("max diff:", diff.max().item())

idx = diff.nonzero()
for i, j in idx[:5]:
    i, j = i.item(), j.item()
    g = j // group_size
    print(f"[{i},{j}] raw_triton={(x_big[i,j]/s_triton[i,g]).item():.6f} raw_ref={(x_big[i,j]/s_ref[i,g]).item():.6f} q_triton={q_triton[i,j].item()} q_ref={q_ref[i,j].item()}")

## 5. 정확도 테스트 (tolerance 기반)

위에서 본 것처럼, 완전 일치 대신 "경계에서만, 드물게, 최대 1 차이"만 허용하도록 짜자.

In [ ]:
def check_correctness():
    torch.manual_seed(0)
    x = torch.randn(37, 256, device="cuda", dtype=torch.float16) * 5
    group_size = 64

    q_triton, s_triton = group_quant_int8_triton(x, group_size)
    q_ref, s_ref = reference_group_quant_int8(x, group_size)

    torch.testing.assert_close(s_triton, s_ref, rtol=1e-4, atol=1e-6)

    diff = (q_triton.int() - q_ref.int()).abs()
    num_mismatched = (diff != 0).sum().item()
    mismatch_ratio = num_mismatched / q_triton.numel()

    assert diff.max().item() <= 1, (
        f"mismatches differ by more than 1 (real bug, not precision noise): "
        f"max diff={diff.max().item()}"
    )
    assert mismatch_ratio < 0.01, (
        f"too many boundary mismatches: {num_mismatched}/{q_triton.numel()}"
    )
    print(
        f"correctness OK: {num_mismatched}/{q_triton.numel()} elements differ "
        "by 1 at integer boundaries (expected fp32 division precision noise)"
    )


check_correctness()

## 6. 밲치마크

PyTorch 순정 구현(reshape → abs → amax → broadcast 나눗셈 → clamp → cast, 여러 개 커널이 순차적으로 실행되며 중간 결과를 매번 GPU 메모리에 씁니다)와, 이 전체를 커널 하나로 합친(fused) Triton 커널을 비교합니다.

In [ ]:
def benchmark():
    x = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
    group_size = 128

    ms_triton = triton.testing.do_bench(lambda: group_quant_int8_triton(x, group_size))
    ms_ref = triton.testing.do_bench(lambda: reference_group_quant_int8(x, group_size))

    print(f"Triton kernel : {ms_triton:.4f} ms")
    print(f"PyTorch ref   : {ms_ref:.4f} ms")
    print(f"speedup       : {ms_ref / ms_triton:.2f}x")


benchmark()

## 7. 면접에서 설명할 수 있는 한 줄 요약

> vLLM의 CUDA quantization 커널을 Triton으로 재구현하면서, 원본과 다른 반올림 규칙 때문에 생긴 정확도 불일치를 디버깅해서 원본의 truncate 방식으로 맞쳤고, 그 과정에서 GPU 나눗셈의 fast-math 근사 오차까지 발견해서 tolerance 기반 테스트로 처리했습니다. 결과적으로 PyTorch 대비 약 4배 빠른 fused 커널을 얻었습니다.